# combine master files

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [1]:
# Install a pip package in the current Jupyter kernel
import importlib, sys, subprocess
print(f"sys.executable: {sys.executable}")
#!{sys.executable} -m pip install numpy==1.22 pillow==8.4 matplotlib==3.2

packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"**** {pkg} module is now installed.")
    else: 
        print(f"******** {pkg} module is already installed.")
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")


sys.executable: /home/guitar79/anaconda3/envs/astro_Python_env/bin/python
******** numpy module is already installed.
******** pandas module is already installed.
******** matplotlib module is already installed.
******** scipy module is already installed.
******** astropy module is already installed.
******** photutils module is already installed.
******** ccdproc module is already installed.
******** version_information module is already installed.
This notebook was generated at 2024-06-09 02:31:29 (KST = GMT+0900) 
0 Python     3.11.5 64bit [GCC 11.2.0]
1 IPython    8.20.0
2 OS         Linux 5.15.0 107 generic x86_64 with glibc2.31
3 numpy      1.26.4
4 pandas     2.2.1
5 matplotlib 3.8.4
6 scipy      1.12.0
7 astropy    6.1.0
8 photutils  1.9.0
9 ccdproc    2.4.2
10 version_information 1.0.4


### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

### import modules

In [2]:
from glob import glob
from pathlib import Path
import os
import shutil
import numpy as np
import astropy.units as u
from astropy.stats import sigma_clip
from ccdproc import combine, ccd_process, CCDData

import ysfitsutilpy as yfu
import ysphotutilpy as ypu
#import ysvisutilpy as yvu

import _astro_utilities
import _Python_utilities

In [3]:
#%%
BASEDIR = Path("/mnt/Rdata/OBS_data") 
DOINGDIR = Path(BASEDIR/ "2024-EXO" / "RiLA600_STX-16803_-_1bin")
#DOINGDIR = Path(BASEDIR/ "2024-EXO" / "GSON300_STF-8300M_-_1bin")

DOINGDIRs = sorted(_Python_utilities.getFullnameListOfsubDirs(DOINGDIR))
DOINGDIRs = sorted([x for x in DOINGDIR.iterdir() if x.is_dir()])
print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

mas1 = [x for x in DOINGDIRs if "CAL-BDF" in str(x)]
mas1 = mas1[0]/ _astro_utilities.master_dir
print ("mas1: ", format(mas1))

DOINGDIRs = sorted([x for x in DOINGDIRs if "_LIGHT_" in str(x)])
print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))
#######################################################

DOINGDIRs:  [PosixPath('/mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/-_CAL-BDF_-_2024-05_-_-_STX-16803_-_1bin'), PosixPath('/mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin'), PosixPath('/mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/TOI-1899B_LIGHT_-_2024-05-21_-_RiLA600_STX-16803_-_1bin'), PosixPath('/mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/TRES-4B_LIGHT_-_2024-05-12_-_RiLA600_STX-16803_-_1bin'), PosixPath('/mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/WASP-135B_LIGHT_-_2024-05-13_-_RiLA600_STX-16803_-_1bin')]
len(DOINGDIRs):  5
mas1:  /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/-_CAL-BDF_-_2024-05_-_-_STX-16803_-_1bin/master_files_ys
DOINGDIRs:  [PosixPath('/mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin'), PosixPath('/mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/TOI-1899B_LIGHT_-_2024-05-21_-_RiLA600_STX-16803_

In [4]:
# import shutil
# mas1 = Path(DOINGDIRs[0]/_astro_utilities.master_dir)

In [5]:
for DOINGDIR in DOINGDIRs[:1] :
    DOINGDIR = Path(DOINGDIR)
    print("DOINGDIR", DOINGDIR)
    fits_in_dir = sorted(list(DOINGDIR.glob('*.fit*')))
    #print("fits_in_dir", fits_in_dir)
    print("len(fits_in_dir)", len(fits_in_dir))

    if len(fits_in_dir) == 0 :
        print(f"There is no fits fils in {DOINGDIR}")
        pass
    else : 
        print(f"Starting: {str(DOINGDIR.parts[-1])}")
    
        MASTERDIR = DOINGDIR / _astro_utilities.master_dir
        REDUCEDDIR = DOINGDIR / _astro_utilities.reduced_dir

        if MASTERDIR.exists():
            shutil.rmtree(MASTERDIR)
        shutil.copytree(mas1, MASTERDIR)

        if not REDUCEDDIR.exists():
            os.makedirs(str(REDUCEDDIR))
            print("{} is created...".format(str(REDUCEDDIR)))

        summary = yfu.make_summary(DOINGDIR/"*.fit*")
        #print(summary)
        print("len(summary):", len(summary))
        print("summary:", summary)
        #print(summary["file"][0])

        df_light = summary.loc[summary["IMAGETYP"] == "LIGHT"].copy()
        df_light = df_light.reset_index(drop=True)

        for _, row in df_light.iterrows():

            fpath = Path(row["file"])
            ccd = yfu.load_ccd(fpath)
            filt = ccd.header["FILTER"]
            expt = ccd.header["EXPTIME"]
            try : 
                red = yfu.ccdred(
                    ccd,
                    output=Path(f"{REDUCEDDIR/ fpath.name}"),
                    mdarkpath=str(MASTERDIR / "master_dark_{:.0f}sec.fits".format(expt)),
                    mflatpath=str(MASTERDIR / "master_flat_{}_norm.fits".format(filt.upper())),
                    # flat_norm_value=1,  # 1 = skip normalization, None = normalize by mean
                    overwrite=True
                )
            except Exception as err :
                print("X"*60)
                #_Python_utilities.write_log(err_log_file, err)

DOINGDIR /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin
len(fits_in_dir) 32
Starting: KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin
/mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/reduced is created...
All 51 keywords (guessed from /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_R_2024-05-23-12-31-00_120sec_RiLA600_STX-16803_25c_1bin.fit) will be loaded.
len(summary): 32
summary:                                                  file  filesize  SIMPLE  \
0   /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803...  33560640    True   
1   /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803...  33560640    True   
2   /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803...  33560640    True   
3   /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803...  33560640    True   
4   /mnt/Rdata/OBS_data/2024-EXO/RiLA60

HISTORY  [yfu.darkcor] Dark subtracted (DARKFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_dark_120sec.fits)
HISTORY  ..................................(dt = 0.035 s) 2024-06-08T17:31:39.985
HISTORY  [yfu.flatcor] Flat pixels with `value < flat_mask = 0` are replaced by `flat_fill = 1`.
HISTORY  .................................................2024-06-08T17:31:40.013
HISTORY  [yfu.flatcor] Flat corrected (FLATFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_flat_R_norm.fits)
HISTORY  ..................................(dt = 0.045 s) 2024-06-08T17:31:40.034
Writing FITS to /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/reduced/KOI-13b_LIGHT_R_2024-05-23-12-31-00_120sec_RiLA600_STX-16803_25c_1bin.fit... Saved.


HISTORY  [yfu.darkcor] Dark subtracted (DARKFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_dark_150sec.fits)
HISTORY  ..................................(dt = 0.035 s) 2024-06-08T17:31:40.987
HISTORY  [yfu.flatcor] Flat pixels with `value < flat_mask = 0` are replaced by `flat_fill = 1`.
HISTORY  .................................................2024-06-08T17:31:41.012
HISTORY  [yfu.flatcor] Flat corrected (FLATFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_flat_R_norm.fits)
HISTORY  ..................................(dt = 0.046 s) 2024-06-08T17:31:41.035
Writing FITS to /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/reduced/KOI-13b_LIGHT_R_2024-05-23-12-33-11_150sec_RiLA600_STX-16803_24c_1bin.fit... Saved.


HISTORY  [yfu.darkcor] Dark subtracted (DARKFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_dark_120sec.fits)
HISTORY  ..................................(dt = 0.036 s) 2024-06-08T17:31:42.268
HISTORY  [yfu.flatcor] Flat pixels with `value < flat_mask = 0` are replaced by `flat_fill = 1`.
HISTORY  .................................................2024-06-08T17:31:42.291
HISTORY  [yfu.flatcor] Flat corrected (FLATFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_flat_R_norm.fits)
HISTORY  ..................................(dt = 0.041 s) 2024-06-08T17:31:42.311
Writing FITS to /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/reduced/KOI-13b_LIGHT_R_2024-05-23-12-41-08_120sec_RiLA600_STX-16803_23c_1bin.fit... Saved.


HISTORY  [yfu.darkcor] Dark subtracted (DARKFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_dark_150sec.fits)
HISTORY  ..................................(dt = 0.036 s) 2024-06-08T17:31:43.576
HISTORY  [yfu.flatcor] Flat pixels with `value < flat_mask = 0` are replaced by `flat_fill = 1`.
HISTORY  .................................................2024-06-08T17:31:43.603
HISTORY  [yfu.flatcor] Flat corrected (FLATFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_flat_R_norm.fits)
HISTORY  ..................................(dt = 0.048 s) 2024-06-08T17:31:43.626
Writing FITS to /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/reduced/KOI-13b_LIGHT_R_2024-05-23-12-43-19_150sec_RiLA600_STX-16803_23c_1bin.fit... Saved.


...

HISTORY  [yfu.darkcor] Dark subtracted (DARKFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_dark_150sec.fits)
HISTORY  ..................................(dt = 0.038 s) 2024-06-08T17:32:10.612
HISTORY  [yfu.flatcor] Flat pixels with `value < flat_mask = 0` are replaced by `flat_fill = 1`.
HISTORY  .................................................2024-06-08T17:32:10.635
HISTORY  [yfu.flatcor] Flat corrected (FLATFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_flat_V_norm.fits)
HISTORY  ..................................(dt = 0.051 s) 2024-06-08T17:32:10.664
Writing FITS to /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/reduced/KOI-13b_LIGHT_V_2024-05-23-13-26-52_150sec_RiLA600_STX-16803_21c_1bin.fit... Saved.


HISTORY  [yfu.darkcor] Dark subtracted (DARKFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_dark_120sec.fits)
HISTORY  ..................................(dt = 0.035 s) 2024-06-08T17:32:11.627
HISTORY  [yfu.flatcor] Flat pixels with `value < flat_mask = 0` are replaced by `flat_fill = 1`.
HISTORY  .................................................2024-06-08T17:32:11.651
HISTORY  [yfu.flatcor] Flat corrected (FLATFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_flat_V_norm.fits)
HISTORY  ..................................(dt = 0.049 s) 2024-06-08T17:32:11.676
Writing FITS to /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/reduced/KOI-13b_LIGHT_V_2024-05-23-13-29-34_120sec_RiLA600_STX-16803_21c_1bin.fit... Saved.


HISTORY  [yfu.darkcor] Dark subtracted (DARKFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_dark_150sec.fits)
HISTORY  ..................................(dt = 0.033 s) 2024-06-08T17:32:12.578
HISTORY  [yfu.flatcor] Flat pixels with `value < flat_mask = 0` are replaced by `flat_fill = 1`.
HISTORY  .................................................2024-06-08T17:32:12.600
HISTORY  [yfu.flatcor] Flat corrected (FLATFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_flat_V_norm.fits)
HISTORY  ..................................(dt = 0.046 s) 2024-06-08T17:32:12.625
Writing FITS to /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/reduced/KOI-13b_LIGHT_V_2024-05-23-13-37-00_150sec_RiLA600_STX-16803_21c_1bin.fit... Saved.


HISTORY  [yfu.darkcor] Dark subtracted (DARKFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_dark_120sec.fits)
HISTORY  ..................................(dt = 0.042 s) 2024-06-08T17:32:13.582
HISTORY  [yfu.flatcor] Flat pixels with `value < flat_mask = 0` are replaced by `flat_fill = 1`.
HISTORY  .................................................2024-06-08T17:32:13.609
HISTORY  [yfu.flatcor] Flat corrected (FLATFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_flat_V_norm.fits)
HISTORY  ..................................(dt = 0.071 s) 2024-06-08T17:32:13.654
Writing FITS to /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/reduced/KOI-13b_LIGHT_V_2024-05-23-13-39-42_120sec_RiLA600_STX-16803_21c_1bin.fit... Saved.


HISTORY  [yfu.darkcor] Dark subtracted (DARKFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_dark_150sec.fits)
HISTORY  ..................................(dt = 0.035 s) 2024-06-08T17:32:14.525
HISTORY  [yfu.flatcor] Flat pixels with `value < flat_mask = 0` are replaced by `flat_fill = 1`.
HISTORY  .................................................2024-06-08T17:32:14.553
HISTORY  [yfu.flatcor] Flat corrected (FLATFRM = /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/master_files_ys/master_flat_V_norm.fits)
HISTORY  ..................................(dt = 0.049 s) 2024-06-08T17:32:14.575
Writing FITS to /mnt/Rdata/OBS_data/2024-EXO/RiLA600_STX-16803_-_1bin/KOI-13b_LIGHT_-_2024-05-23_-_RiLA600_STX-16803_-_1bin/reduced/KOI-13b_LIGHT_V_2024-05-23-13-47-08_150sec_RiLA600_STX-16803_21c_1bin.fit... Saved.
